In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.compose import TransformedTargetRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut, cross_val_predict
from sklearn.callback import ProgressBar
from sklearn import set_config
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

set_config(enable_metadata_routing=True)

### Read data

In [ ]:
climate = pd.read_parquet("phenocam_site_in_usa_climate.parquet")
climate.columns = ["year", "yday", "prcp", "tmax", "tmin", "vp", "site"]
pheno   = pd.read_parquet("phenocam_phenology_model.parquet")
site_meta = pd.read_parquet("phenocam_site_in_usa.parquet")

### Summarize meteorology

We construct precipitation and temperature time series at daily and monthly temporal resolution. 

In [ ]:
climate_monthly = climate\
    .assign(month = lambda x: "month" + ((x.yday // 30)+1).astype(str))\
    .groupby(["site", "year", "month"])[["prcp", "tmax"]]\
    .agg("mean")\
    .reset_index()\
    .pivot(index=["site", "year"], columns="month")

climate_monthly.columns = ['_'.join(col).strip() for col in climate_monthly.columns.values]

climate_monthly.head()

In [ ]:
# Convert climate to wide format - essentially make the full prcp/tmax time series
# the columns
climate_daily = climate\
    .assign(yday=lambda x: "yday" + x.yday.astype(str))\
    .drop(columns=["tmin", "vp"])\
    .pivot(index=["site", "year"], columns="yday")
# Clean column names
climate_daily.columns = ['_'.join(col).strip() for col in climate_daily.columns.values]
climate_daily.head()

### Join phenological transitions

In [ ]:
# Join phenological transitions onto site-years
climate_pheno_monthly = pd.merge(
    climate_monthly,
    pheno[["site", "transition_year_rising", "veg_type", "doy_transition_50_rising"]],
    how="inner",
    left_on=["site", "year"],
    right_on=["site", "transition_year_rising"]
)

climate_pheno_daily = pd.merge(
    climate_daily,
    pheno[["site", "transition_year_rising", "veg_type", "doy_transition_50_rising"]],
    how="inner",
    left_on=["site", "year"],
    right_on=["site", "transition_year_rising"]
)

### PLSR model

Structure this as a transformed target regressor to account for angular response.

In [ ]:
class VonMisesTransformer(BaseEstimator, TransformerMixin):
    '''
    Implements an angular transformation of the target variable. This transformation
    is used in, for example:

    https://bg.copernicus.org/articles/17/3991/2020/
    '''
    def __init__(self, cycle_length=365):
        self.cycle_length = cycle_length
        self.to_rad_ = (2 * np.pi) / cycle_length
        self.mean_angle_ = None

    def to_radians(self, x):
        return x * self.to_rad_

    def from_radians(self, x):
        return x / self.to_rad_
        
    def fit(self, X, y=None):
        self.mean_angle_ = np.mean(self.to_radians(X))
        return self
        
    def transform(self, X, y=None):
        return np.tan((self.to_radians(X) - self.mean_angle_) / 2)

    def inverse_transform(self, X, y=None):
        return self.from_radians(2 * np.atan(X) + self.mean_angle_)

For scoring, use J-S correlation coefficient to account for wrapping.

In [ ]:
def jammalamadaka_sharma_corr(alpha, beta, cycle_length=365):
    """
    Computes the Jammalamadaka-Sharma circular correlation coefficient.
    """
    alpha = np.asarray(alpha)
    beta = np.asarray(beta)

    alpha = alpha * (2 * np.pi) / cycle_length
    beta  = beta * (2 * np.pi) / cycle_length
    
    # Compute circular means using mean of sines and cosines
    mean_alpha = np.arctan2(np.mean(np.sin(alpha)), np.mean(np.cos(alpha)))
    mean_beta = np.arctan2(np.mean(np.sin(beta)), np.mean(np.cos(beta)))
    
    # Compute deviations
    sin_alpha_dev = np.sin(alpha - mean_alpha)
    sin_beta_dev = np.sin(beta - mean_beta)
    
    # Calculate numerator and denominator components
    num = np.sum(sin_alpha_dev * sin_beta_dev)
    den = np.sqrt(np.sum(sin_alpha_dev**2) * np.sum(sin_beta_dev**2))
    
    if den == 0:
        return 0.0
    return num / den

js_scorer = make_scorer(jammalamadaka_sharma_corr)

### Grid-search for N components

In [ ]:
import warnings
warnings.filterwarnings(
    "ignore", message=".*The estimator TransformedTargetRegressor does not support callbacks.*"
)

In [ ]:
cv_daily = GridSearchCV(
    TransformedTargetRegressor(
        regressor=PLSRegression(),
        transformer=VonMisesTransformer(cycle_length=365),
        check_inverse=False
    ),
    param_grid = dict(regressor__n_components = list(range(1, 10))),
    scoring = js_scorer,
    cv=LeaveOneGroupOut()
)

cv_daily.set_callbacks(ProgressBar())

X_daily = climate_pheno_daily.filter(regex="(tmax|prcp)")
y_daily = climate_pheno_daily["doy_transition_50_rising"]

cv_daily.fit(X_daily, y_daily, groups=climate_pheno_daily["site"])

In [ ]:
print("Best parameters")
print(cv_daily.best_params_)

print("JS Score on full dataset")
print(js_scorer(cv_daily, X_daily, y_daily))

Repeat at monthly timestep. Hold components at 4 for comparability.

In [ ]:
cv_monthly = GridSearchCV(
    TransformedTargetRegressor(
        regressor=PLSRegression(),
        transformer=VonMisesTransformer(cycle_length=365),
        check_inverse=False
    ),
    param_grid = dict(regressor__n_components = [4]),
    scoring = js_scorer,
    cv=LeaveOneGroupOut()
)

cv_monthly.set_callbacks(ProgressBar())

X_monthly = climate_pheno_monthly.filter(regex="(tmax|prcp)")
y_monthly = climate_pheno_monthly["doy_transition_50_rising"]

cv_monthly.fit(X_monthly, y_monthly, groups=climate_pheno_monthly["site"])

In [ ]:
print("Best parameters")
print(cv_monthly.best_params_)

print("JS Score on full dataset")
print(js_scorer(cv_monthly, X_monthly, y_monthly))

### Generate out-of-sample predictions

In [ ]:
climate_pheno_daily["model_doy_transition_50_rising"] = cross_val_predict(
    cv_daily.estimator, 
    X_daily, 
    y_daily, 
    cv=cv_daily.cv, 
    params=dict(groups=climate_pheno_daily["site"])
)

climate_pheno_monthly["model_doy_transition_50_rising"] = cross_val_predict(
    cv_monthly.estimator, 
    X_monthly, 
    y_monthly, 
    cv=cv_monthly.cv, 
    params=dict(groups=climate_pheno_monthly["site"])
)

### Aggregate MAE

In [ ]:
def angular_absolute_error(y_true, y_hat, cycle_length=365):
    to_rad = (2 * np.pi) / cycle_length
    
    y_true_rad = y_true * to_rad
    y_hat_rad  = y_hat  * to_rad

    error_rad = np.minimum(np.abs(y_true_rad - y_hat_rad), 2*np.pi - np.abs(y_true_rad - y_hat_rad))

    return error_rad / to_rad

climate_pheno_daily["model_error_doy"] = angular_absolute_error(
    climate_pheno_daily["doy_transition_50_rising"],
    climate_pheno_daily["model_doy_transition_50_rising"]
)

climate_pheno_monthly["model_error_doy"] = angular_absolute_error(
    climate_pheno_monthly["doy_transition_50_rising"],
    climate_pheno_monthly["model_doy_transition_50_rising"]
)

In [ ]:
daily_error_agg = climate_pheno_daily\
    .groupby("site")\
    .agg(dict(model_error_doy=["mean", "count"]))\
    .droplevel(0, axis=1)\
    .join(site_meta[["MAT_daymet", "MAP_daymet"]])

monthly_error_agg = climate_pheno_monthly\
    .groupby("site")\
    .agg(dict(model_error_doy=["mean", "count"]))\
    .droplevel(0, axis=1)\
    .join(site_meta[["MAT_daymet", "MAP_daymet"]])

Within-site average error over climate space.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, sharex=False, figsize=(8, 4))

size_gain = 4
monthly_error_agg["count_gain"] = monthly_error_agg["count"] * size_gain
daily_error_agg["count_gain"] = daily_error_agg["count"] * size_gain

plot_args = dict(
    kind="scatter",
    x="MAT_daymet", 
    y="MAP_daymet", 
    c="mean", 
    s="count_gain",
    edgecolors="black",
    vmin=0, 
    vmax=90, 
    colorbar=False,
    linewidth=0.5
)

daily_error_agg.plot(
    ax=ax1,
    **plot_args
)

monthly_error_agg.plot(
    ax=ax2,
    **plot_args
)

for ax in (ax1, ax2):
    ax.grid()
    ax.set_xlabel("")
    ax.set_ylabel("")

cbar = plt.colorbar(ax1.collections[0], ax=(ax1, ax2), orientation='vertical', fraction=.1)
cbar.set_label("Average site error (days)")

ax1.set_title("Daily timesteps")
ax2.set_title("Monthly timesteps")

fig.supxlabel("MAT (deg C)")
fig.supylabel("MAP (mm)")

In [ ]:
pd.merge(
    daily_error_agg,
    monthly_error_agg,
    left_index=True,
    right_index=True,
    suffixes=["_daily", "_monthly"]
).plot(x="mean_daily", y="mean_monthly", kind="scatter", s="count_gain_daily", edgecolor="black", linewidth=0.5)

plt.grid()
plt.xlabel("Daily timestep MAE")
plt.ylabel("Monthly timestep MAE")
plt.axline((0, 0), (1, 1), color="black")

### Distinct models by veg type

Now that we know the appropriate time step and number of components to use, we can see if there is much of a benefit to training distinct PLSR models on each vegetation type. This might improve the greater error in hot/dry areas seen above.

In [ ]:
climate_pheno_monthly.groupby("veg_type")["site"].agg("count")

In [ ]:
MIN_OBS = 100
ESTIMATOR_ARGS = dict(
    n_components=4
)

estimators = {}

cv_predict_results = []

for veg_type, sub_df in climate_pheno_monthly.groupby("veg_type"):
    if sub_df.shape[0] < MIN_OBS:
        print(f"{veg_type} has fewer than {MIN_OBS} observations. Fitting global model.")
        model_df = climate_pheno_monthly.copy()
    else:
        model_df = sub_df.copy()

    X_sub = model_df.filter(regex="(tmax|prcp)")
    y_sub = model_df["doy_transition_50_rising"]
    
    # Fit estimator
    estimator = TransformedTargetRegressor(
        regressor=PLSRegression(**ESTIMATOR_ARGS),
        transformer=VonMisesTransformer(cycle_length=365),
        check_inverse=False
    ).fit(X_sub, y_sub)
    estimators[veg_type] = estimator
    
    # Run LOO predictions by site    
    y_sub_hat = cross_val_predict(
        estimator,
        X_sub,
        y_sub,
        cv=LeaveOneGroupOut(),
        params=dict(groups=model_df["site"])
    )

    model_df["model_cv_doy_transition_50_rising"] = y_sub_hat

    cv_predict_results.append(model_df)

Global performance metrics.

In [ ]:
cv_predict = pd.concat(cv_predict_results, axis=0)
cv_predict["model_error_doy"] = angular_absolute_error(cv_predict["doy_transition_50_rising"], cv_predict["model_cv_doy_transition_50_rising"])

print(f"Global mean angular absolute error: {cv_predict["model_error_doy"].mean():.2f}")
print(f"Global JS correlation: {jammalamadaka_sharma_corr(cv_predict["doy_transition_50_rising"], cv_predict["model_cv_doy_transition_50_rising"]):.2f}")

In [ ]:
monthly_cv_error_agg = cv_predict\
    .groupby("site")\
    .agg(dict(model_error_doy=["mean", "count"]))\
    .droplevel(0, axis=1)\
    .join(site_meta[["MAT_daymet", "MAP_daymet"]])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, sharex=False, figsize=(8, 4))

size_gain = 4
monthly_error_agg["count_gain"] = monthly_error_agg["count"] * size_gain
monthly_cv_error_agg["count_gain"] = daily_error_agg["count"] * size_gain

plot_args = dict(
    kind="scatter",
    x="MAT_daymet", 
    y="MAP_daymet", 
    c="mean", 
    s="count_gain",
    edgecolors="black",
    vmin=0, 
    vmax=90, 
    colorbar=False,
    linewidth=0.5
)

monthly_cv_error_agg.plot(
    ax=ax1,
    **plot_args
)

monthly_error_agg.plot(
    ax=ax2,
    **plot_args
)

for ax in (ax1, ax2):
    ax.grid()
    ax.set_xlabel("")
    ax.set_ylabel("")

cbar = plt.colorbar(ax1.collections[0], ax=(ax1, ax2), orientation='vertical', fraction=.1)
cbar.set_label("Average site error (days)")

ax1.set_title("Separate models by veg type")
ax2.set_title("Single model")

fig.supxlabel("MAT (deg C)")
fig.supylabel("MAP (mm)")

In [ ]:
pd.merge(
    monthly_cv_error_agg,
    monthly_error_agg,
    left_index=True,
    right_index=True,
    suffixes=["_cv_monthly", "_monthly"]
).plot(x="mean_cv_monthly", y="mean_monthly", kind="scatter", s="count_gain_monthly", edgecolor="black", linewidth=0.5)

plt.grid()
plt.xlabel("Multiple model MAE")
plt.ylabel("Single model MAE")
plt.axline((0, 0), (1, 1), color="black")

There doesn't seem to be evidence in favor of separate models for each vegetation type.